# UZIMA Fabric Data Access

This notebook reads the ODBC connection string from Key Vault.

The connection string uses the `cdiofabric` managed identity to connect to Fabric.

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
import pandas as pd
import pyodbc

vault = SecretClient(
    vault_url="https://uzima-fabric-tokens.vault.azure.net/",
    credential=DefaultAzureCredential(),
)

connection_string = vault.get_secret("fabric-odbc-connection-string").value

# To use another database, change only the database name.
# connection_string = connection_string.replace("DATABASE=uzima_db_backup;", "DATABASE=Fitbit;")

connection = pyodbc.connect(connection_string, timeout=30)

## See Available Tables

In [ ]:
tables = pd.read_sql(
    """
    SELECT TABLE_SCHEMA, TABLE_NAME
    FROM INFORMATION_SCHEMA.TABLES
    ORDER BY TABLE_SCHEMA, TABLE_NAME
    """,
    connection,
)

tables.head(30)

## Read A Small Sample

After you see the table list, change `table_to_read` to the approved table or masked view you need.

In [ ]:
table_to_read = "dbo.dimenrolledparticipants"

sample = pd.read_sql(f"SELECT TOP 10 * FROM {table_to_read}", connection)
sample

In [ ]:
connection.close()